In [ ]:

from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

project_dir = Path('/content/drive/MyDrive/CSE499-Medical-AI')
dataset_dir = project_dir / '01_Dataset/CheXpert/CheXpert-v1.0-small'

print("Project folder exists:", project_dir.exists())
print("Dataset folder exists:", dataset_dir.exists())
print("Training CSV exists:", (dataset_dir / 'train.csv').exists())
print("Validation CSV exists:", (dataset_dir / 'valid.csv').exists())


##Load and inspect the CheXpert CSV files

In [ ]:

import pandas as pd

train_df = pd.read_csv(dataset_dir / 'train.csv')
valid_df = pd.read_csv(dataset_dir / 'valid.csv')

print("Training dataset shape:", train_df.shape)
print("Validation dataset shape:", valid_df.shape)

print("\nColumn names:")
print(train_df.columns.tolist())

print("\nFirst 5 training records:")
display(train_df.head())


##Analyze missing values

In [ ]:

# Count missing values in each column
train_missing = train_df.isna().sum()
valid_missing = valid_df.isna().sum()

# Calculate missing percentages
missing_report = pd.DataFrame({
    "Train Missing": train_missing,
    "Train Missing (%)": (train_missing / len(train_df) * 100).round(2),
    "Valid Missing": valid_missing,
    "Valid Missing (%)": (valid_missing / len(valid_df) * 100).round(2)
})

print("CheXpert Missing Value Report:")
display(missing_report)

print("\nTotal missing entries in training CSV:", int(train_missing.sum()))
print("Total missing entries in validation CSV:", int(valid_missing.sum()))


##Analyze all four label categories

Now we will count positive, negative, uncertain, and missing entries for each of the 14 medical findings.

In [ ]:

# Select the 14 medical finding columns
label_columns = train_df.columns[5:]

# Count each label category in the training dataset
label_report = pd.DataFrame({
    "Positive (1)": (train_df[label_columns] == 1).sum(),
    "Negative (0)": (train_df[label_columns] == 0).sum(),
    "Uncertain (-1)": (train_df[label_columns] == -1).sum(),
    "Missing (NaN)": train_df[label_columns].isna().sum()
})

print("CheXpert Training Label Analysis:")
display(label_report)

print("\nTotal positive:", int(label_report["Positive (1)"].sum()))
print("Total negative:", int(label_report["Negative (0)"].sum()))
print("Total uncertain:", int(label_report["Uncertain (-1)"].sum()))
print("Total missing:", int(label_report["Missing (NaN)"].sum()))

print("\nAll labels accounted for:",
      int(label_report.to_numpy().sum()) == len(train_df) * len(label_columns))


##Analyze validation labels

We have analyzed the training labels. Now we need to understand the validation labels, because we will eventually use them to evaluate our deep learning models.

In [ ]:

# Count label categories in the validation dataset
valid_label_report = pd.DataFrame({
    "Positive (1)": (valid_df[label_columns] == 1).sum(),
    "Negative (0)": (valid_df[label_columns] == 0).sum(),
    "Uncertain (-1)": (valid_df[label_columns] == -1).sum(),
    "Missing (NaN)": valid_df[label_columns].isna().sum()
})

print("CheXpert Validation Label Analysis:")
display(valid_label_report)

print("\nTotal positive:", int(valid_label_report["Positive (1)"].sum()))
print("Total negative:", int(valid_label_report["Negative (0)"].sum()))
print("Total uncertain:", int(valid_label_report["Uncertain (-1)"].sum()))
print("Total missing:", int(valid_label_report["Missing (NaN)"].sum()))

print("\nAll labels accounted for:",
      int(valid_label_report.to_numpy().sum()) == len(valid_df) * len(label_columns))


##Analyze patient demographics and X-ray views

Now we will examine the age, sex, and image-view distributions in the training and validation datasets.

In [ ]:

# Compare patient metadata and X-ray view distributions
for name, df in [("Training", train_df), ("Validation", valid_df)]:
    print(f"\n{name} Dataset")

    print("\nSex distribution:")
    print(df["Sex"].value_counts(dropna=False))

    print("\nAge statistics:")
    print(df["Age"].describe().round(2))

    print("\nX-ray view distribution:")
    print(df["Frontal/Lateral"].value_counts(dropna=False))

    print("\nAP/PA distribution:")
    print(df["AP/PA"].value_counts(dropna=False))


##Analyze X-ray image dimensions

We will inspect a reproducible sample of 100 training X-rays to understand their image dimensions and file formats.

In [ ]:

from PIL import Image
import pandas as pd

# Select a reproducible sample of 100 training images
sample_df = train_df.sample(n=100, random_state=42)

image_info = []

for _, row in sample_df.iterrows():
    image_path = dataset_dir.parent / row["Path"]

    try:
        with Image.open(image_path) as img:
            image_info.append({
                "Width": img.width,
                "Height": img.height,
                "Format": img.format,
                "Mode": img.mode
            })
    except Exception as e:
        print("Image error:", image_path, str(e))

image_info_df = pd.DataFrame(image_info)

print("Images successfully analyzed:", len(image_info_df))

if not image_info_df.empty:
    print("\nImage dimensions:")
    display(image_info_df[["Width", "Height"]].describe().round(2))

    print("\nImage formats:")
    print(image_info_df["Format"].value_counts())

    print("\nImage color modes:")
    print(image_info_df["Mode"].value_counts())


###Analyze X-ray image intensity

Now we will examine the pixel intensity values of the same 100 X-ray images.

This helps us understand image brightness and contrast before normalization.

In [ ]:

import numpy as np

intensity_results = []

for _, row in sample_df.iterrows():
    image_path = dataset_dir.parent / row["Path"]

    try:
        with Image.open(image_path) as img:
            pixels = np.array(img.convert("L"))

            intensity_results.append({
                "Mean": pixels.mean(),
                "Std": pixels.std(),
                "Min": pixels.min(),
                "Max": pixels.max()
            })
    except Exception as e:
        print("Image error:", image_path, str(e))

intensity_df = pd.DataFrame(intensity_results)

print("Images analyzed:", len(intensity_df))

print("\nPixel Intensity Statistics:")
display(intensity_df.describe().round(2))

print("\nAverage image brightness:", round(intensity_df["Mean"].mean(), 2))
print("Average image contrast (pixel std):", round(intensity_df["Std"].mean(), 2))


##Visualize sample chest X-ray images

We will display six randomly selected chest X-rays from our training dataset and save the figure to Google Drive.

In [ ]:

import matplotlib.pyplot as plt
from PIL import Image

# Select 6 random X-ray records from the training dataset
sample_images = train_df.sample(n=6, random_state=42)

# Create a figure with 2 rows and 3 columns
fig, axes = plt.subplots(2, 3, figsize=(12, 9))

# Go through the 6 selected X-ray records
for ax, (_, row) in zip(axes.flat, sample_images.iterrows()):

    # Build the complete image path in Google Drive
    image_path = dataset_dir.parent / row["Path"]

    # Open the X-ray image and convert it to grayscale
    with Image.open(image_path) as img:
        image = img.convert("L")

        # Display the image using a grayscale color map
        ax.imshow(image, cmap="gray")

    # Show the image view and patient age above each image
    ax.set_title(f'{row["Frontal/Lateral"]} | Age: {row["Age"]}')

    # Hide the X and Y axes for a cleaner figure
    ax.axis("off")

# Add a title to the complete figure
fig.suptitle("CheXpert Sample Chest X-ray Images", fontsize=16)

# Adjust spacing so images and titles do not overlap
plt.tight_layout()

# Create the figures folder if it does not already exist
figures_dir = project_dir / "05_Figures"
figures_dir.mkdir(parents=True, exist_ok=True)

# Save the figure to Google Drive
figure_path = figures_dir / "chexpert_sample_xrays.png"
plt.savefig(figure_path, dpi=150, bbox_inches="tight")

# Display the completed figure
plt.show()

# Confirm that the figure was saved
print("Figure saved:", figure_path.exists())
print("Figure location:", figure_path)


##erification using 1,000 images


In [ ]:

%%time

import time

# Record the starting time
start_time = time.time()

# Select 1,000 random training records
sample_train = train_df.sample(n=1000, random_state=42)

# Count missing image files
train_missing = 0
valid_missing = 0

# Check the 1,000 sampled training images
for image_path in sample_train["Path"]:

    # Build the complete image path
    full_path = dataset_dir.parent / image_path

    # Count the file if it does not exist
    if not full_path.is_file():
        train_missing += 1

# Check all 234 validation images
for image_path in valid_df["Path"]:

    # Build the complete image path
    full_path = dataset_dir.parent / image_path

    # Count the file if it does not exist
    if not full_path.is_file():
        valid_missing += 1

# Display the verification results
print("Training images sampled:", len(sample_train))
print("Training images missing:", train_missing)

print("\nValidation images checked:", len(valid_df))
print("Validation images missing:", valid_missing)

# Confirm whether all checked images exist
sample_verified = train_missing == 0 and valid_missing == 0

print("\nSample verification passed:", sample_verified)

# Calculate total execution time
elapsed_time = time.time() - start_time

print("Execution time in seconds:", round(elapsed_time, 2))
print("Execution time in minutes:", round(elapsed_time / 60, 2))


##Save our dataset verification report

Now we will save the results from STEP 3 through STEP 11 into a JSON report in Google Drive.

In [ ]:

import json
from datetime import datetime, timezone

# Choose where to save our research report
report_dir = project_dir / "00_Project_Documents"
report_dir.mkdir(parents=True, exist_ok=True)

# Collect verified results from previous steps
report = {
    "project": "Medical-AI-CXR",
    "notebook": "02_dataset_verification.ipynb",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),

    "dataset": {
        "training_records": len(train_df),
        "validation_records": len(valid_df),
        "columns": len(train_df.columns),
        "medical_labels": len(label_columns)
    },

    "training_labels": {
        "positive": int(label_report["Positive (1)"].sum()),
        "negative": int(label_report["Negative (0)"].sum()),
        "uncertain": int(label_report["Uncertain (-1)"].sum()),
        "missing": int(label_report["Missing (NaN)"].sum())
    },

    "validation_labels": {
        "positive": int(valid_label_report["Positive (1)"].sum()),
        "negative": int(valid_label_report["Negative (0)"].sum()),
        "uncertain": int(valid_label_report["Uncertain (-1)"].sum()),
        "missing": int(valid_label_report["Missing (NaN)"].sum())
    },

    "sample_image_analysis": {
        "images_analyzed": len(image_info_df),
        "average_width": round(float(image_info_df["Width"].mean()), 2),
        "average_height": round(float(image_info_df["Height"].mean()), 2),
        "average_brightness": round(float(intensity_df["Mean"].mean()), 2),
        "average_pixel_std": round(float(intensity_df["Std"].mean()), 2)
    },

    "image_path_verification": {
        "training_paths_sampled": len(sample_train),
        "training_paths_missing": train_missing,
        "validation_paths_checked": len(valid_df),
        "validation_paths_missing": valid_missing,
        "full_training_scan_completed": False,
        "sample_verification_passed": bool(sample_verified),
        "execution_seconds": round(elapsed_time, 2)
    }
}

# Save the report as a readable JSON file
report_path = report_dir / "serial02_dataset_report.json"

with open(report_path, "w", encoding="utf-8") as file:
    json.dump(report, file, indent=4)

# Confirm the report was saved
print("Report saved:", report_path.exists())
print("Report location:", report_path)
print("Report size:", report_path.stat().st_size, "bytes")


##Verify the saved JSON report

In [ ]:

import json

# Open the saved dataset verification report
with open(report_path, "r", encoding="utf-8") as file:
    saved_report = json.load(file)

# Display the saved report
print(json.dumps(saved_report, indent=4))

# Verify the important dataset statistics
assert saved_report["dataset"]["training_records"] == 223414
assert saved_report["dataset"]["validation_records"] == 234
assert saved_report["dataset"]["medical_labels"] == 14

# Confirm the sampled image verification results
assert saved_report["image_path_verification"]["training_paths_sampled"] == 1000
assert saved_report["image_path_verification"]["training_paths_missing"] == 0
assert saved_report["image_path_verification"]["validation_paths_missing"] == 0
assert saved_report["image_path_verification"]["full_training_scan_completed"] is False

# Print success only if all checks pass
print("\nDataset report verification: PASSED")


##Full CheXpert image verification



In [ ]:

import json
import time
from pathlib import Path
from PIL import Image

# Combine training and validation image paths
all_paths = (
    train_df["Path"].tolist()
    + valid_df["Path"].tolist()
)

# Set the checkpoint and final report locations
checkpoint_path = project_dir / "00_Project_Documents/full_image_check_progress.json"
final_report_path = project_dir / "00_Project_Documents/full_image_verification.json"

# Load saved progress if a previous run was interrupted
if checkpoint_path.exists():
    with open(checkpoint_path, "r") as file:
        progress = json.load(file)
else:
    progress = {
        "checked": 0,
        "valid": 0,
        "missing": 0,
        "unreadable": 0,
        "errors": []
    }

# Record the start time of this run
start_time = time.time()

print("Total image records:", len(all_paths))
print("Resuming from image:", progress["checked"])

# Continue checking from the saved position
for i in range(progress["checked"], len(all_paths)):

    # Build the full Google Drive image path
    image_path = dataset_dir.parent / all_paths[i]

    try:
        # Open and verify that the image can be decoded
        with Image.open(image_path) as img:
            img.verify()

        progress["valid"] += 1

    except FileNotFoundError:
        # Record missing image files
        progress["missing"] += 1
        progress["errors"].append({
            "index": i,
            "path": all_paths[i],
            "error": "File not found"
        })

    except Exception as e:
        # Record images that cannot be read
        progress["unreadable"] += 1
        progress["errors"].append({
            "index": i,
            "path": all_paths[i],
            "error": str(e)
        })

    # Update the number of checked images
    progress["checked"] = i + 1

    # Save progress after every 500 images
    if progress["checked"] % 500 == 0:

        # Write to a temporary file before replacing checkpoint
        temp_path = checkpoint_path.with_suffix(".tmp")

        with open(temp_path, "w") as file:
            json.dump(progress, file, indent=2)

        temp_path.replace(checkpoint_path)

        # Display progress and elapsed time
        elapsed = (time.time() - start_time) / 60

        print(
            f'Checked: {progress["checked"]}/{len(all_paths)} | '
            f'Valid: {progress["valid"]} | '
            f'Missing: {progress["missing"]} | '
            f'Unreadable: {progress["unreadable"]} | '
            f'Elapsed: {elapsed:.1f} min',
            flush=True
        )

# Save the final verification report
with open(final_report_path, "w") as file:
    json.dump(progress, file, indent=2)

# Save the final checkpoint too
with open(checkpoint_path, "w") as file:
    json.dump(progress, file, indent=2)

print("\nFULL IMAGE VERIFICATION COMPLETED")
print("Total checked:", progress["checked"])
print("Valid images:", progress["valid"])
print("Missing images:", progress["missing"])
print("Unreadable images:", progress["unreadable"])
print("Report saved:", final_report_path)


In [ ]:

from google.colab import drive
from pathlib import Path
import pandas as pd
import json

# Reconnect Google Drive
drive.mount("/content/drive", force_remount=False)

# Restore project and dataset locations
project_dir = Path("/content/drive/MyDrive/CSE499-Medical-AI")
dataset_dir = project_dir / "01_Dataset/CheXpert/CheXpert-v1.0-small"

# Reload training and validation CSV files
train_df = pd.read_csv(dataset_dir / "train.csv")
valid_df = pd.read_csv(dataset_dir / "valid.csv")

# Rebuild the original image list in the same order
all_paths = train_df["Path"].tolist() + valid_df["Path"].tolist()

# Read the previously saved checkpoint
checkpoint_path = project_dir / "00_Project_Documents/full_image_check_progress.json"

print("Training records:", len(train_df))
print("Validation records:", len(valid_df))
print("Total image records:", len(all_paths))

if checkpoint_path.exists():
    with open(checkpoint_path, "r") as file:
        progress = json.load(file)

    print("\nCheckpoint recovered successfully")
    print("Images checked:", progress["checked"])
    print("Valid images:", progress["valid"])
    print("Missing images:", progress["missing"])
    print("Unreadable images:", progress["unreadable"])
else:
    print("\nCheckpoint not found. Do not restart the scan.")


In [ ]:

from pathlib import Path
import json
import time

# Locate the project folder
project_dir = Path("/content/drive/MyDrive/CSE499-Medical-AI")

# Locate the saved checkpoint
checkpoint_path = project_dir / "00_Project_Documents/full_image_check_progress.json"

# Read only the small checkpoint file
start = time.time()

if checkpoint_path.exists():
    with open(checkpoint_path, "r") as file:
        progress = json.load(file)

    print("Checkpoint recovered:", True)
    print("Images checked:", progress["checked"])
    print("Valid images:", progress["valid"])
    print("Missing images:", progress["missing"])
    print("Unreadable images:", progress["unreadable"])
else:
    print("Checkpoint not accessible")

print("Execution time:", round(time.time() - start, 2), "seconds")


In [ ]:

import pandas as pd

# Restore the dataset location
dataset_dir = project_dir / "01_Dataset/CheXpert/CheXpert-v1.0-small"

# Load the original CSV files
train_df = pd.read_csv(dataset_dir / "train.csv")
valid_df = pd.read_csv(dataset_dir / "valid.csv")

# Rebuild the image list in its original order
all_paths = train_df["Path"].tolist() + valid_df["Path"].tolist()

# Confirm the checkpoint matches the dataset
assert len(all_paths) == 223648
assert progress["checked"] == (
    progress["valid"] + progress["missing"] + progress["unreadable"]
)

print("Training records:", len(train_df))
print("Validation records:", len(valid_df))
print("Resume from image:", progress["checked"])
print("Dataset ready:", True)


In [ ]:

from google.colab import drive
from pathlib import Path
import pandas as pd
import json

# Reconnect Google Drive
drive.mount("/content/drive", force_remount=False)

# Restore project and dataset locations
project_dir = Path("/content/drive/MyDrive/CSE499-Medical-AI")
dataset_dir = project_dir / "01_Dataset/CheXpert/CheXpert-v1.0-small"

# Recover the saved verification checkpoint
checkpoint_path = project_dir / "00_Project_Documents/full_image_check_progress.json"

with open(checkpoint_path, "r") as file:
    progress = json.load(file)

# Show recovered progress before loading CSV files
print("Checkpoint recovered:", True)
print("Images checked:", progress["checked"])
print("Valid images:", progress["valid"])
print("Missing images:", progress["missing"])
print("Unreadable images:", progress["unreadable"])

# Reload the original dataset CSV files
train_df = pd.read_csv(dataset_dir / "train.csv")
valid_df = pd.read_csv(dataset_dir / "valid.csv")

# Rebuild the image list in its original order
all_paths = train_df["Path"].tolist() + valid_df["Path"].tolist()

# Confirm the recovered progress is consistent
assert len(all_paths) == 223648
assert progress["checked"] == (
    progress["valid"] + progress["missing"] + progress["unreadable"]
)

print("\nDataset ready:", True)
print("Resume from image:", progress["checked"])


In [ ]:

from google.colab import drive
from pathlib import Path
import json
import pandas as pd

drive.mount("/content/drive")

project_dir = Path("/content/drive/MyDrive/CSE499-Medical-AI")
dataset_dir = project_dir / "01_Dataset/CheXpert/CheXpert-v1.0-small"

checkpoint_path = project_dir / "00_Project_Documents/full_image_check_progress.json"

with open(checkpoint_path, "r") as file:
    progress = json.load(file)

train_df = pd.read_csv(dataset_dir / "train.csv")
valid_df = pd.read_csv(dataset_dir / "valid.csv")

all_paths = train_df["Path"].tolist() + valid_df["Path"].tolist()

assert len(all_paths) == 223648
assert progress["checked"] == (
    progress["valid"] + progress["missing"] + progress["unreadable"]
)

print("Checkpoint recovered:", True)
print("Images checked:", progress["checked"])
print("Valid images:", progress["valid"])
print("Missing images:", progress["missing"])
print("Unreadable images:", progress["unreadable"])
print("Images remaining:", len(all_paths) - progress["checked"])
print("Dataset ready:", True)


##Investigate the four unreadable images

In [ ]:

from google.colab import drive
from pathlib import Path
import json
from PIL import Image

# Connect Google Drive
drive.mount("/content/drive")

# Restore project locations
project_dir = Path("/content/drive/MyDrive/CSE499-Medical-AI")
dataset_dir = project_dir / "01_Dataset/CheXpert/CheXpert-v1.0-small"

# Load the completed verification report
report_path = project_dir / "00_Project_Documents/full_image_verification.json"

with open(report_path, "r") as file:
    report = json.load(file)

print("Report loaded successfully:", True)
print("Total checked:", report["checked"])
print("Valid images:", report["valid"])
print("Unreadable images:", report["unreadable"])
print("Recorded errors:", len(report["errors"]))


In [ ]:

# Retry the four images that failed verification
for item in report["errors"]:
    image_path = dataset_dir.parent / item["path"]

    print("\nImage index:", item["index"])
    print("Original error:", item["error"])

    try:
        with Image.open(image_path) as img:
            img.load()
            print("Retry result: SUCCESS")
            print("Image size:", img.size)
            print("Image mode:", img.mode)
    except Exception as e:
        print("Retry result: FAILED")
        print("New error:", str(e))


In [ ]:

# Check the two images that still failed
for item in report["errors"]:
    if item["index"] not in [181000, 181001]:
        continue

    image_path = dataset_dir.parent / item["path"]

    print("\nImage index:", item["index"])
    print("File exists:", image_path.exists())

    try:
        print("File size:", image_path.stat().st_size, "bytes")
    except OSError as e:
        print("Metadata error:", str(e))


##Test direct file reading

In [ ]:

# Test direct reading of the two remaining files
for item in report["errors"]:
    if item["index"] not in [181000, 181001]:
        continue

    image_path = dataset_dir.parent / item["path"]

    print("\nImage index:", item["index"])

    try:
        with open(image_path, "rb") as file:
            header = file.read(16)

        print("Read status: SUCCESS")
        print("Bytes read:", len(header))
        print("JPEG signature:", header[:3].hex())

    except OSError as e:
        print("Read status: FAILED")
        print("Error:", str(e))


##Verify both images completely

In [ ]:

from PIL import Image

# Fully decode the two remaining images
for item in report["errors"]:
    if item["index"] not in [181000, 181001]:
        continue

    image_path = dataset_dir.parent / item["path"]

    print("\nImage index:", item["index"])

    try:
        with Image.open(image_path) as img:
            img.load()
            print("Full decoding: SUCCESS")
            print("Image size:", img.size)
            print("Image mode:", img.mode)
    except Exception as e:
        print("Full decoding: FAILED")
        print("Error:", str(e))


##Save the final verification summary

In [ ]:

import json
from datetime import datetime, timezone

# Preserve the original verification report
final_summary = {
    "dataset": "CheXpert-v1.0-small",
    "total_images": report["checked"],
    "initial_scan_valid": report["valid"],
    "initial_scan_unreadable": report["unreadable"],
    "initial_scan_missing": report["missing"],
    "failed_images_retested": 4,
    "successful_retries": 4,
    "unresolved_failures": 0,
    "retry_method": "PIL Image.open and Image.load",
    "original_error_type": "Google Drive Errno 5 input/output error",
    "verification_status": "COMPLETED",
    "completed_at_utc": datetime.now(timezone.utc).isoformat()
}

summary_path = (
    project_dir
    / "00_Project_Documents"
    / "serial02_final_verification_summary.json"
)

with open(summary_path, "w") as file:
    json.dump(final_summary, file, indent=2)

print("Final summary saved:", summary_path.exists())
print("Total images:", final_summary["total_images"])
print("Successful retries:", final_summary["successful_retries"])
print("Unresolved failures:", final_summary["unresolved_failures"])
print("Verification status:", final_summary["verification_status"])


##Prepare the GitHub notebook

In [ ]:

from pathlib import Path

# Locate the completed notebook
notebook_path = (
    project_dir
    / "02_Notebooks"
    / "02_dataset_verification.ipynb"
)

# Locate the research reports
documents_dir = project_dir / "00_Project_Documents"

files_to_check = {
    "Dataset verification notebook": notebook_path,
    "Dataset report": documents_dir / "serial02_dataset_report.json",
    "Full image verification": documents_dir / "full_image_verification.json",
    "Final verification summary": documents_dir / "serial02_final_verification_summary.json"
}

# Check whether each file exists
for name, path in files_to_check.items():
    print(name, ":", "FOUND" if path.is_file() else "MISSING")
